# **Sequências usando RNNs e CNNs**

## **Motivação**

Redes feedforward passam informação em uma única direção: input → camadas → output. Isso funciona bem quando os dados são **independentes** entre si (ex: classificar imagens), mas falha quando existe **dependência sequencial**, prever x sem o contexto de x-1, x-2... é uma desvantagem.

**Exemplo:** em "O moço viu um carro e não atravessou a ___", só sabemos que a resposta é "rua" com o contexto das palavras anteriores e além disso se uma palavra ali troca de posicao tudo muda.

*Vamos usar todo o passado como input entao?*  Sequências têm tamanhos variáveis e passados longos tornam o input inviável.

*resumir o passado com estatísticas (média, moda, mediana...) ?*
Funciona, mas tem limitações:
- **Perde a ordem** dos eventos (ex: [1,2,10] e [10,2,1] têm a mesma média, mas são sequências opostas) 
- Informacao muito antiga pode **enviesar as estatisticas**
- Exige **feature engineering manual** (descobrir na mão o que importa)

→ Troca **capacidade de captar padrões complexos** por **simplicidade/interpretabilidade**.

Ai entra RNNs que criam uma **memória dos inputs anteriores**, armazenada como parâmetros da rede, aprendida automaticamente, sem feature engineering manual. Elas trabalham com sequencias, se antes uma instancia/exemplo era representado por *m* colunas de features, agora uma instancia é representada por *m* vetores (sequencias temporais) de tamanho max_len, ou seja, *m* caracteristicas e suas evoluções e *max_len* timesteps.

## **Neuronios recorrentes e camadas**

A entrada de uma camada recorrente tem sempre 3 dimensões sendo elas [seq_len, batch_size, n_features] por padrão do pytorch e [batch_size, seq_len, n_features] caso o parametro *batch_first = True*. Sendo cada uma dessas dimensões:

- **batch_size:** tamanho do batch da epoca (quantas instancias considerar para cada atualizacao dos parametros em uma epoca)
    - Um exemplo seria 12 clientes computados por vez para computar o gradiente
- **seq_len:** tamanho maximo das sequencias de uma instancia, ou seja, tamanho maximo de passos no tempo
    - Um exemplo seria 12 meses/passos de histórico de cada cliente
    - Em sequencias de tamanho variado, uma vez definido o max_len, todas as sequencias que forem menores que max_len são completadas com padding que são valores artificiais que permitem todas as instancias caberem em um unico tensor comportado. A *função pack_padded_sequence(...)* é uma técnica de otimização que informa à LSTM exatamente onde cada sequência termina, para que ela ignore o padding e não "aprenda" com dados falsos.
- **n_features (input_size):** quantas variaveis existem a cada passo de tempo 
    - Um exemplo seria 3 caracteristicas do cliente em cada mes como gastos, chamadas e usos

Dentro dessa camada existem *hidden_size* neurônios, cada um deles recebe 2 vetores de entrada e tem 2 vetores de pesos:

- ${x_t}$: Vetor de features de uma instancia no tempo t, ou seja, os valores das series temporais no tempo t de dimensão = ${1 * n_{input}}$. Ao concatenar todas as instancias do mini-batch temos uma matriz ${X_t}$ de dimensao ${n_{batch} * n_{input}}$
- ${y_{t-1}}$: Vetor de saidas do timestep anterior (uma saida por neuronio) de dimensão = ${1 * n_{neuronios}}$. Ao concater a saida de todas as instancias do batch temos uma matriz ${Y_{t-1}}$ de dimensao ${n_{batch} * n_{neuronios}}$
- ${w_x}$: Vetor de pesos do input de dimensão = ${1 * n_{input}}$. Ao concatenar verticalmente os vetores de pesos de todos os neuronios temos uma matriz ${W_x}$ de dimensao ${n_{neuronios} * n_{input}}$
- ${w_{y_{t-1}}}$:Vetor de pesos do timestep anterior de dimensão = ${1 * n_{neuronios}}$. Ao concatenar veticalmente os pesos de todos os neuronios temos uma matriz ${W_{y_{t-1}}}$ de dimensao ${n_{neuronios} * n_{n_neuronios}}$

A saida de um timestep t é definida como:

- Para uma unica instancia x no tempo t:

$$
\begin{equation}
\mathbf{y}_{(t)} = \phi\left( \mathbf{W}_x^T \mathbf{x}_{(t)} + \mathbf{W}_{\hat{y}}^T \hat{\mathbf{y}}_{(t-1)} + \mathbf{b} \right)
\end{equation}
$$

- Para uma mini-batch inteiro no tempo t (as matrizes ainda podem ser concatenadas e feitas em uma unica operacao):

$$
\begin{equation}
\hat{\mathbf{Y}}_{(t)} = \phi\left( \mathbf{X}_{(t)} \mathbf{W}_x + \hat{\mathbf{Y}}_{(t-1)} \mathbf{W}_{\hat{y}} + \mathbf{b} \right)
\end{equation}
$$

Uma instancia de treino do batch é um conjunto de n_features que sao vetores de tamanho seq_len (timesteps). Para cada camada recorrente as sequencias sao entregues um timestep por vez (todas as n_features daquele timestep) do menor para o maior, ou seja, 0 -> 1 -> ...-> seq_len (no timestep 0, o output anterior geralmente é zerado).

### **Celulas de memoria**

Como a saida de um ou mais neuronios no timestep t é uma funcao de todos os timesteps anteriore, chamamos isso de celulas de memória. Uma celula com apenas um ou mais neuronios é a forma mais simples possivel de RNN e por isso tem uma capacidade limitada de aprendizado temporal (uns 10 timesteps em media antes de perder qualidade), existem camadas mais complexas como GRU e LSTM que extendem essa memória. O estado de uma celula no tempo t é chamado de ${h_t}$ que é uma funcao ${f(x_t , h_{t-1})}$, para a RNN mais ismples esse estado é simplesmente a saida do timestep mas em camadas mais complexas isso nao é mais verdade.

### **Input e Output**

A entrada e saida de camadas recorrentes podem variar muito dependendo do objetivo:

- **sequence to sequence:** cada timestep tem uma saida individual ${y_{t}}$, a loss é computada usando a saida de todos os timestamps. Util para coisas como: prever t+1 dado os t anteriores (isso poderia ser modelado apenas gerando o ultimo t mas fazendo desse jeito temos mais sinal de erro)
- **sequence to vector:** mesma sequência de entrada, mas agora a rede ignora as saídas intermediárias, só a última importa. É o caso do sentiment analysis: você lê a sequência inteira, e só no final decide "bom" ou "ruim".
- **vector to sequence:** entra com um único vetor (repetido em cada timestep) e a rede gera uma sequência inteira. É o caso de image captioning: uma imagem vira um vetor (via CNN), e esse vetor "alimenta" a RNN em todos os passos pra gerar uma legenda palavra por palavra.
- **encoder-decoder:** primeiro um encoder (seq-to-vector) comprime a sequência de entrada num único vetor, depois um decoder (vector-to-sequence) usa esse vetor pra gerar a sequência de saída. É o caso de tradução em que ler toda a frase primeiro é importante.

![image.png](imagens/rnn_tipos.png)

## **Treino de RNNs**

O estado no timestep t depende de todos os estados anteriores a t, ou seja, é uma função composta, da mesma forma que em redes densas a saída final nada mais é que uma composição das saídas das camadas anteriores. A diferença chave é que, numa RNN, essa composição acontece no tempo, não em profundidade: é como se cada timestep fosse "desenrolado" (unrolled) numa camada própria, mas compartilhando os mesmos pesos (Wx, Wŷ, b) em todos os passos. Podemos usar Backpropagation Through Time (BPTT) para atualizar os pesos, levando em consideração essa dependência temporal. O processo tem duas etapas:

- *Forward pass:* a rede processa a sequência inteira, timestep a timestep, gerando uma saída y(t) em cada passo.
- *Backward pass:* como todas as saídas dependem (direta ou indiretamente) dos mesmos pesos compartilhados, o gradiente de cada timestep se acumula, soma-se o gradiente de todos os passos antes de fazer uma única atualização dos pesos.

Dependendo da tarefa, nem todos os outputs dos timesteps entram na loss. Mas, mesmo quando só a última saída importa, o gradiente ainda precisa percorrer toda a cadeia de timesteps anteriores (porque y(N) depende de y(N-1), que depende de y(N-2)...), então o custo computacional do BPTT continua proporcional ao comprimento da sequência Além disso, para sequências muito longas, BPTT completo fica caro e sofre de vanishing/exploding gradients (o gradiente precisa atravessar muitos passos multiplicativos). Na prática, usa-se truncated BPTT, limita-se o número de timesteps que o gradiente propaga pra trás, mesmo que a sequência de entrada seja mais longa.